In [ ]:
import os
import pandas as pd
import numpy as np
from plotly import graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import to_html
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


%load_ext autoreload
%autoreload 2


In [ ]:
# Load pile_aggregate and eval_aggregate data
# pile_losses = pd.read_csv('pile_aggregate_n44.csv', index_col=0)
# eval_losses = pd.read_csv('evals_aggregate_n60.csv', index_col=0)
pile_losses = pd.read_csv('pile_aggregate_n58.csv', index_col=0)
eval_losses = pd.read_csv('evals_aggregate_n58.csv', index_col=0)
print(f"Original Pile losses shape: {pile_losses.shape}")
print(f"Original Eval losses shape: {eval_losses.shape}")

# Restrict both dfs to the same rows
common_indexes = pile_losses.index.intersection(eval_losses.index)
pile_losses = pile_losses.loc[common_indexes]
eval_losses = eval_losses.loc[common_indexes]
print(f"Pile losses shape: {pile_losses.shape}")
print(f"Eval losses shape: {eval_losses.shape}")

figure_path = "figures/aggregate_pca_v1"

In [ ]:
pile_small = pd.read_csv('pile_aggregate_n44.csv', index_col=0)
# which indexes of pile_losses are not in pile_small?
missing_indexes = pile_losses.index.difference(pile_small.index)
print(missing_indexes)

In [ ]:
from aggregate_pca import PCAAnalysis
normalise = True
n_components = 10
pca_X = PCAAnalysis(pile_losses, name="Pile", normalise=normalise, n_components=n_components)
pca_Y = PCAAnalysis(eval_losses, name="Evals", normalise=normalise, n_components=n_components)

# Also apply PCA to both datasets without outliers (NO= No Outliers)
### NOTE: There are likely some new outliers from the new n58 data
outlier_models = ['eleutherai/pythia-14m','eleutherai/pythia-31m',
                  'eleutherai/pythia-70m','eleutherai/pythia-160m', 
                  'eleutherai/pythia-410m',
                  'google/gemma-7b-it', 'google/gemma-2b-it']
pca_X_NO = PCAAnalysis(pile_losses[~pile_losses.index.isin(outlier_models)], name="Pile (NO)", normalise=normalise, n_components=n_components)
pca_Y_NO = PCAAnalysis(eval_losses[~eval_losses.index.isin(outlier_models)], name="Evals (NO)", normalise=normalise, n_components=n_components)

In [ ]:
from aggregate_pca import plot_pc_score_heatmaps
fig1 = plot_pc_score_heatmaps(pca_X, pca_Y)
fig1.show()
fig1.write_image(f"{figure_path}/pc_score_heatmaps.png", scale=3)

In [ ]:
from aggregate_pca import plot_score_correlation_heatmap

fig_corr = plot_score_correlation_heatmap(pca_X, pca_Y)
fig_corr.show()
fig_corr.write_image(f"{figure_path}/pc_score_correlations.png", scale=3)

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig2 = plot_pc_loading_heatmap(pca_Y)
fig2.show()
fig2.write_image(f"{figure_path}/pc_eval_loadings.png", scale=3)

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig3 = plot_pc_loading_heatmap(pca_X)
fig3.show()
fig3.write_image(f"{figure_path}/pc_pile_loadings_contexts.png", scale=2)

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig4 = plot_pc_loading_heatmap(pca_X, mean_loadings=True)
fig4.show()
fig4.write_image(f"{figure_path}/pc_pile_loadings_means.png", scale=3)

In [ ]:
from aggregate_pca import calculate_dk_values, plot_dk_dict, calculate_dk_dict
dk_values = calculate_dk_dict(pca_X, pca_Y, normalise=True)
fig_dk = plot_dk_dict(dk_values, pca_X_name="Pile", pca_Y_name="Evals")
fig_dk.show()
fig_dk.write_image(f"{figure_path}/Dk_maxkx10_maxky10.png", scale=3)

# No outliers

In [ ]:
from aggregate_pca import plot_pc_score_heatmaps
fig1 = plot_pc_score_heatmaps(pca_X_NO, pca_Y_NO)
fig1.show()
fig1.write_image(f"{figure_path}/pc_score_heatmaps_NO.png", scale=3)

In [ ]:
from aggregate_pca import plot_score_correlation_heatmap

fig_corr = plot_score_correlation_heatmap(pca_X_NO, pca_Y_NO)
fig_corr.show()
fig_corr.write_image(f"{figure_path}/pc_score_correlations_NO.png", scale=3)

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig2 = plot_pc_loading_heatmap(pca_Y_NO)
fig2.show()
fig2.write_image(f"{figure_path}/pc_eval_loadings_NO.png", scale=3)

In [ ]:
from datasets import load_dataset
pile_subsets_mini = load_dataset("timaeus/pile_subsets_mini", split="train")


In [ ]:
print(pile_subsets_mini[575]['subset'])
print(list(pile_subsets_mini[575]['text']))

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig3 = plot_pc_loading_heatmap(pca_X_NO)
fig3.show()
fig3.write_image(f"{figure_path}/pc_pile_loadings_contexts_NO.png", scale=2)

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig4 = plot_pc_loading_heatmap(pca_X_NO, mean_loadings=True)
fig4.show()
fig4.write_image(f"{figure_path}/pc_pile_loadings_means_NO.png", scale=3)

In [ ]:
from aggregate_pca import calculate_dk_values, plot_dk_dict, calculate_dk_dict
dk_values = calculate_dk_dict(pca_X_NO, pca_Y_NO, normalise=True)
fig_dk = plot_dk_dict(dk_values, pca_X_name="Pile (NO)", pca_Y_name="Evals (NO)")
fig_dk.show()
fig_dk.write_image(f"{figure_path}/Dk_maxkx10_maxky10_NO.png", scale=3)

# Attempt logistic transformation

In [ ]:
from aggregate_pca import PCAAnalysis
normalise = True
n_components = 10

epsilon= 1e-06
pile_transformed = np.log(pile_losses + epsilon)  
# apply relu to all numbers
eval_relu = np.clip(eval_losses.drop(['cola', 'squadv2'],axis=1), 0, None)  # Ensure no negative values
eval_adjusted = (eval_relu + epsilon) / (1 + 2*epsilon)
eval_transformed = np.log(eval_adjusted / (1 - eval_adjusted))


pca_X = PCAAnalysis(pile_transformed, name="Pile", normalise=normalise, n_components=n_components)
pca_Y = PCAAnalysis(eval_transformed, name="Evals", normalise=normalise, n_components=n_components)

# Also apply PCA to both datasets without outliers (NO= No Outliers)
### NOTE: There are likely some new outliers from the new n58 data
outlier_models = ['eleutherai/pythia-14m','eleutherai/pythia-31m',
                  'eleutherai/pythia-70m','eleutherai/pythia-160m', 
                  'eleutherai/pythia-140m',
                  'google/gemma-7b-it', 'google/gemma-2b-it']
pca_X_NO = PCAAnalysis(pile_transformed[~pile_transformed.index.isin(outlier_models)], name="Pile (NO)", normalise=normalise, n_components=n_components)
pca_Y_NO = PCAAnalysis(eval_transformed[~eval_transformed.index.isin(outlier_models)], name="Evals (NO)", normalise=normalise, n_components=n_components)

In [ ]:
pile_transformed.head()

In [ ]:
eval_transformed.head()

In [ ]:
# apply relu to all numbers
eval_relu = np.clip(eval_losses.drop(['cola', 'squadv2'],axis=1), 0, None)  # Ensure no negative values
eval_adjusted = (eval_relu + epsilon) / (1 + 2*epsilon)
eval_transformed = np.log(eval_adjusted / (1 - eval_adjusted))
# print number of nans in eval_transformed
print(f"Number of NaNs in eval_transformed: {np.isnan(eval_transformed).sum().sum()}")

In [ ]:
# get all columns that are not bound to [0,1]
for col in eval_losses.columns:
    if not ((eval_losses[col] >= 0) & (eval_losses[col] <= 1)).all():
        print(f"Column {col} is not bound to [0,1]")

In [ ]:
from aggregate_pca import plot_pc_score_heatmaps
fig1 = plot_pc_score_heatmaps(pca_X_NO, pca_Y_NO)
fig1.show()
#fig1.write_image(f"{figure_path}/pc_score_heatmaps_NO.png", scale=3)

In [ ]:
from aggregate_pca import calculate_dk_values, plot_dk_dict, calculate_dk_dict
dk_values = calculate_dk_dict(pca_X_NO, pca_Y_NO, normalise=True)
fig_dk = plot_dk_dict(dk_values, pca_X_name="Pile (log)", pca_Y_name="Evals (logit)")
fig_dk.show()

In [ ]:
eval_losses['cola']

In [ ]:
eval_losses['squadv2']

In [ ]:
# plot squadv2 against cola in eval_losses in plotly express
import plotly.express as px
fig = px.scatter(eval_losses, x='cola', y='squadv2', title='Squad v2 vs CoLA Losses',
                 labels={'cola': 'CoLA Loss', 'squadv2': 'Squad v2 Loss'},
                 hover_name=eval_losses.index)
fig.show()

In [ ]:
display(eval_losses.loc[['01-ai/yi-6b']])